## Helpers

In [0]:
CATALOG = "workspace"
critical_failures = []
warnings = []

def critical(name, passed, detail=""):
    print(("PASS" if passed else "FAIL"), "-", name, detail)
    if not passed: critical_failures.append(name)

def warning(name, passed, detail=""):
    print(("PASS" if passed else "WARN"), "-", name, detail)
    if not passed: warnings.append(name)

def count(sql):
    return spark.sql(sql).collect()[0][0]

## Critical checks

In [0]:
critical("customers: no duplicate customer_id",
         count(f"SELECT COUNT(*) FROM (SELECT customer_id FROM {CATALOG}.silver.crm_customers GROUP BY customer_id HAVING COUNT(*) > 1)") == 0)

critical("products: every product has exactly one current row",
         count(f"""SELECT COUNT(*) FROM (
                       SELECT product_number FROM {CATALOG}.silver.crm_products
                       GROUP BY product_number
                       HAVING SUM(CASE WHEN end_date IS NULL THEN 1 ELSE 0 END) <> 1
                   )""") == 0)

critical("sales: every customer_id exists in customers",
         count(f"""SELECT COUNT(*) FROM {CATALOG}.silver.crm_sales s
                   LEFT JOIN {CATALOG}.silver.crm_customers c ON s.customer_id = c.customer_id
                   WHERE c.customer_id IS NULL""") == 0)

critical("sales: every product_number exists as a current product",
         count(f"""SELECT COUNT(*) FROM {CATALOG}.silver.crm_sales s
                   LEFT JOIN (SELECT product_number FROM {CATALOG}.silver.crm_products WHERE end_date IS NULL) p
                          ON s.product_number = p.product_number
                   WHERE p.product_number IS NULL""") == 0)

critical("sales: sales_amount = quantity x price",
         count(f"SELECT COUNT(*) FROM {CATALOG}.silver.crm_sales WHERE sales_amount <> quantity * price") == 0)

## Result

In [0]:
print(f"\nCritical failures: {len(critical_failures)} | Warnings: {len(warnings)}")
assert not critical_failures, f"Silver checks FAILED: {critical_failures}. Gold will NOT be built."
print("Gate passed.")